# Exploring rotation CSV data

Two CSV files live in `data/exp/rotation_csvs/`. Each was exported from an Excel workbook tracking neuroblast (NB) division axes over time. This notebook:

1. Parses the files into a tidy DataFrame, stripping the Excel garbage
2. Explains what the angle columns actually measure and verifies it geometrically
3. Flags 3 NBs with copy-paste errors in the spreadsheet and recomputes correct angles from raw coordinates
4. Computes rotation axes and signed XY-plane rotation (the missing azimuthal information)
5. Derives simulation parameters and checks whether the step distribution is self-consistent with the cumulative trajectories

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pathlib import Path

DATA_DIR = Path("../data/exp/rotation_csvs")

## 1. Parse the CSVs

The files are messy:
- The header spans **2 physical lines** because one column name is a quoted multi-line string — we skip both with `skiprows=2`
- The data has **33 columns**; only the first 27 are meaningful (6 trailing empty columns exist)
- Each NB group starts with a row that has the filename/genotype/NB id; subsequent rows leave those blank → forward-fill
- After real data rows come ~10 padding rows filled with `#DIV/0!`, `###...###`, `00:00:00` — a real row has a numeric integer in `time_point`

In [ ]:
COLS = [
    "file_name", "hours_AEL", "time_scale", "genotype", "NB_id",
    "time_point", "time_hms", "delta_time",
    "apical_x", "apical_y", "apical_z",
    "basal_x",  "basal_y",  "basal_z",
    "dx", "dy", "dz",
    "spindle_length",
    "dot_product_3d",
    "mag_t",
    "mag_t1",
    "mag_product",
    "cos_angle",
    "angle_from_first_deg",
    "direction_correction",
    "angle_from_prev_deg",
    "abs_angle_from_prev_deg",
]

def load_csv(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(
        path,
        skiprows=2,
        header=None,
        names=COLS + [f"_x{i}" for i in range(6)],
        dtype=str,
        na_values=["#DIV/0!", "#REF!", "FALSE", "TRUE", ""],
        keep_default_na=True,
    )[COLS]

    id_cols = ["file_name", "hours_AEL", "time_scale", "genotype", "NB_id"]
    raw[id_cols] = raw[id_cols].ffill()

    raw["time_point"] = pd.to_numeric(raw["time_point"], errors="coerce")
    raw = raw[raw["time_point"].notna()].copy()

    for c in [
        "apical_x", "apical_y", "apical_z",
        "basal_x",  "basal_y",  "basal_z",
        "dx", "dy", "dz",
        "spindle_length", "dot_product_3d",
        "mag_t", "mag_t1", "mag_product", "cos_angle",
        "angle_from_first_deg", "angle_from_prev_deg", "abs_angle_from_prev_deg",
    ]:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

    raw["time_point"] = raw["time_point"].astype(int)
    raw["source_file"] = path.stem
    return raw.reset_index(drop=True)


dfs = [load_csv(p) for p in sorted(DATA_DIR.glob("*.csv"))]
df = pd.concat(dfs, ignore_index=True)

print(f"{len(df)} rows, {df['source_file'].nunique()} files")
print(f"Genotypes: {sorted(df['genotype'].dropna().unique())}")
print(f"Total NB recordings: {len(df.groupby(['source_file', 'file_name', 'NB_id']))}")

In [ ]:
nb_table = (
    df.groupby(["source_file", "file_name", "genotype", "NB_id"])
    .agg(n_timepoints=("time_point", "count"), max_tp=("time_point", "max"))
    .reset_index()
)
nb_table

## 2. What do the angle columns actually measure?

### The division axis vector

At each time point the apical and basal pole positions are recorded in pixels. The division axis vector is:

$$\vec{v}(t) = (\Delta x,\; \Delta y,\; \Delta z) = \text{basal} - \text{apical}$$

This is a **full 3-D vector** (despite the misleading column header `Sqrt(Δx²+Δy²)`, the spreadsheet stored the 3-D magnitude — verified below).

### `angle_from_first_deg` — the 3-D angle from the initial spindle

$$\theta(t) = \arccos\!\left(\frac{\vec{v}(t)\cdot\vec{v}(t_1)}{|\vec{v}(t)|\;|\vec{v}(t_1)|}\right) \in [0°,\;180°]$$

### `angle_from_prev_deg` — step-to-step change in cumulative angle

$$\Delta\theta(t) = \theta(t) - \theta(t-1)$$

### Why the sign is ambiguous

$\arccos$ always returns values in $[0°, 180°]$, discarding which side of the cone the vector landed on. `angle_from_prev_deg` can be negative simply because the NB drifted back toward its initial axis — not because it rotated in any defined direction.

## 3. Verify the formula and find spreadsheet errors

In [ ]:
def recompute_angle_from_first(group: pd.DataFrame) -> pd.Series:
    row1 = group[group["time_point"] == 1]
    if row1.empty:
        return pd.Series(np.nan, index=group.index)
    v1 = row1.iloc[0][["dx", "dy", "dz"]].values.astype(float)
    m1 = np.linalg.norm(v1)
    if m1 == 0 or np.isnan(v1).any():
        return pd.Series(np.nan, index=group.index)
    out = []
    for _, row in group.iterrows():
        vt = row[["dx", "dy", "dz"]].values.astype(float)
        mt = np.linalg.norm(vt)
        if mt == 0 or np.isnan(vt).any():
            out.append(np.nan)
        else:
            cos_a = np.clip(np.dot(vt, v1) / (mt * m1), -1.0, 1.0)
            out.append(np.degrees(np.arccos(cos_a)))
    return pd.Series(out, index=group.index)


df["angle_recomputed"] = (
    df.groupby(["source_file", "file_name", "NB_id"], group_keys=False)
    .apply(recompute_angle_from_first)
)

check = df[df["angle_from_first_deg"].notna() & df["angle_recomputed"].notna()].copy()
check["residual"] = (check["angle_recomputed"] - check["angle_from_first_deg"]).abs()

print(f"Rows compared: {len(check)}")
print(f"Within 1°: {(check['residual'] < 1).sum()}")
print(f"Diff > 1° (spreadsheet errors): {(check['residual'] >= 1).sum()}")

check[check["residual"] >= 1][["file_name", "NB_id", "time_point",
                                "angle_from_first_deg", "angle_recomputed",
                                "residual", "mag_t1"]]

144 rows match within 1°, confirming the 3-D interpretation. The 5 mismatching rows (3 NBs) have copy-paste errors in the spreadsheet: `mag_t1` and `dot_product_3d` referenced the wrong NB's cells. **We use `angle_recomputed` from here on.**

In [ ]:
def delta_angle(group):
    g = group.sort_values("time_point")
    return pd.Series(g["angle_recomputed"].diff().values, index=g.index)

df["delta_angle_recomputed"] = (
    df.groupby(["source_file", "file_name", "NB_id"], group_keys=False)
    .apply(delta_angle)
)

## 4. Initial exploration of the 3-D angles

In [ ]:
genotypes = sorted(df["genotype"].dropna().unique())
angle_df = df[df["time_point"] > 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in [
    (axes[0], "angle_recomputed",       "Cumulative 3-D angle from first axis (°)"),
    (axes[1], "delta_angle_recomputed", "Δ 3-D angle from previous time point (°)"),
]:
    for geno, grp in angle_df.groupby("genotype"):
        ax.hist(grp[col].dropna(), bins=20, alpha=0.6, label=f"{geno} (n={grp[col].notna().sum()})")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(genotypes), figsize=(7 * len(genotypes), 5), sharey=True)
if len(genotypes) == 1:
    axes = [axes]
for ax, geno in zip(axes, genotypes):
    sub = df[df["genotype"] == geno]
    for (fn, nb_id), grp in sub.groupby(["file_name", "NB_id"]):
        g = grp.sort_values("time_point")
        valid = g[g["angle_recomputed"].notna()]
        if len(valid) > 1:
            ax.plot(valid["time_point"], valid["angle_recomputed"],
                    marker="o", alpha=0.55, linewidth=1.2)
    ax.set_title(geno)
    ax.set_xlabel("Time point")
    ax.set_ylabel("3-D angle from first axis (°)")
    ax.axhline(0, color="k", linewidth=0.5, linestyle="--")
fig.suptitle("Each line = one NB recording")
plt.tight_layout()
plt.show()

## 5. Summary statistics

In [ ]:
angle_df.groupby("genotype")[["angle_recomputed", "delta_angle_recomputed"]].agg(
    ["count", "mean", "median", "std"]
)

## 6. Rotation axis and signed rotation in the imaging plane

### The missing piece: what the cross product adds

The `arccos` angle places v(t) *somewhere* on a cone of half-angle θ around v(t₁) — but says nothing about where on that cone. The missing piece is the **rotation axis**: the axis around which v(t₁) needs to rotate to reach v(t) along the shortest arc.

This axis is given by the cross product:

$$\hat{n}(t) = \frac{\vec{v}(t_1) \times \vec{v}(t)}{|\vec{v}(t_1) \times \vec{v}(t)|}$$

By the right-hand rule, $\hat{n}$ points in the direction such that if your right thumb points along $\hat{n}$, your fingers curl from v(t₁) toward v(t). Together with θ, this fully specifies the rotation (this is called the **axis-angle representation**).

The z-component of $\hat{n}$ is especially interpretable:
- **n_z > 0**: the rotation looks **counter-clockwise** when viewed from above (+z)
- **n_z < 0**: **clockwise** from above
- **|n_z| ≈ 0**: the spindle tilted mostly out of the imaging plane (rotation around an in-plane axis)

We compute two variants:
- **n_cum**: cumulative rotation axis from v(t₁) to v(t) — encodes the azimuthal position on the cone
- **n_step**: rotation axis for each individual step, v(t−1) → v(t)

### Why the rotation axis is potentially interesting

If NB spindles have a preferred rotation direction — say, they consistently rotate around the anterior-posterior axis of the embryo — n_step would cluster in a particular direction on the unit sphere rather than being uniformly distributed. A non-uniform distribution of n_step is evidence for a biologically constrained rotation direction, not just a random diffusion of spindle orientation.

Comparing n_step distributions between wild-type and *mud4* could reveal whether the mutation changes the **direction** of rotation, not just its magnitude.

### Caveat: image coordinate system

Without knowing how each embryo was oriented in the microscope, we cannot equate the image z-axis with any anatomical axis. The rotation axis directions are interpretable within each recording's frame but are only comparable across recordings if embryos were consistently mounted.

### XY-plane signed angle — the 2D analog

A simpler and directly 2D-relevant quantity is the signed angle of the spindle **projected into the imaging plane**:

$$\varphi(t) = \operatorname{atan2}(\Delta y,\; \Delta x) \in (-180°,\; 180°]$$

The **signed step rotation in XY** is:

$$\Delta\varphi(t) = \varphi(t) - \varphi(t-1) \quad \text{wrapped to } (-180°, 180°]$$

Positive = CCW, negative = CW, viewed from +z. This is a genuine signed angle — unlike `delta_angle_recomputed` which comes from arccos and has ambiguous sign. It is the quantity most directly analogous to what a 2D simulation models, though it discards any rotation that tilts the spindle in z.

**Note on large values:** any `|Δφ| > ~150°` is suspicious — it likely reflects an apical/basal labeling swap between time points rather than a genuine near-180° rotation in one step. These rows are flagged below.

In [ ]:
# Compute rotation axes (cumulative and step) and XY-plane angles
records = {}
for (sf, fn, nb), grp in df.groupby(["source_file", "file_name", "NB_id"]):
    g = grp.sort_values("time_point")
    rows = list(g.iterrows())
    v1 = rows[0][1][["dx", "dy", "dz"]].values.astype(float)
    phi_prev = np.degrees(np.arctan2(v1[1], v1[0]))
    v_prev = v1.copy()

    for i, (idx, row) in enumerate(rows):
        vt = row[["dx", "dy", "dz"]].values.astype(float)
        rec = {}
        if not np.isnan(vt).any():
            phi_t = np.degrees(np.arctan2(vt[1], vt[0]))
            rec["phi_xy"] = phi_t
            if i > 0 and not np.isnan(phi_prev):
                rec["delta_phi_xy"] = ((phi_t - phi_prev) + 180) % 360 - 180
                # cumulative rotation axis v(t1) -> v(t)
                n = np.cross(v1, vt)
                nm = np.linalg.norm(n)
                if nm > 1e-10:
                    n /= nm
                    rec.update({"n_cum_x": n[0], "n_cum_y": n[1], "n_cum_z": n[2]})
                # step rotation axis v(t-1) -> v(t)
                ns = np.cross(v_prev, vt)
                nsm = np.linalg.norm(ns)
                if nsm > 1e-10:
                    ns /= nsm
                    rec.update({"n_step_x": ns[0], "n_step_y": ns[1], "n_step_z": ns[2]})
            phi_prev = phi_t
        v_prev = vt.copy()
        records[idx] = rec

df = df.join(pd.DataFrame.from_dict(records, orient="index"))

# Cumulative XY rotation from t1 = running sum of signed steps
def cum_phi(group):
    g = group.sort_values("time_point")
    return pd.Series(g["delta_phi_xy"].fillna(0).cumsum().values, index=g.index)

df["cum_phi_xy"] = df.groupby(["source_file", "file_name", "NB_id"], group_keys=False).apply(cum_phi)

# Flag suspicious large steps
suspicious = df[df["delta_phi_xy"].abs() > 150][["file_name", "NB_id", "time_point", "delta_phi_xy", "genotype"]]
if len(suspicious):
    print("Rows with |Δφ_xy| > 150° — probable apical/basal labeling swap:")
    display(suspicious)

In [ ]:
step_df = df[df["time_point"] > 1].copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# --- Row 0: rotation axis direction ---

# n_step_z histogram — tells us CW vs CCW preference
ax = axes[0, 0]
for geno, grp in step_df.groupby("genotype"):
    ax.hist(grp["n_step_z"].dropna(), bins=15, alpha=0.6, label=geno)
ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("n_step_z  (+ = CCW from above,  − = CW from above)")
ax.set_ylabel("Count")
ax.set_title("Step rotation axis: z-component\n(tells you CW vs CCW in the imaging plane)")
ax.legend()

# n_step scatter (x vs y, colored by z) — where on the sphere do rotation axes cluster?
for col_idx, geno in enumerate(genotypes):
    ax = axes[0, col_idx + 1]
    sub = step_df[(step_df["genotype"] == geno) & step_df["n_step_x"].notna()]
    sc = ax.scatter(sub["n_step_x"], sub["n_step_y"], c=sub["n_step_z"],
                    cmap="coolwarm", vmin=-1, vmax=1, s=45, alpha=0.75)
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), color="gray", linewidth=0.8, alpha=0.4)
    ax.set_aspect("equal")
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_xlabel("n_step_x"); ax.set_ylabel("n_step_y")
    ax.set_title(f"{geno}: step rotation axis (XY projection)\ncolor = n_step_z (red=CCW, blue=CW)")
    plt.colorbar(sc, ax=ax, label="n_step_z", shrink=0.8)

# --- Row 1: XY-plane signed rotation ---

# Histogram of delta_phi_xy
ax = axes[1, 0]
for geno, grp in step_df.groupby("genotype"):
    ax.hist(grp["delta_phi_xy"].dropna(), bins=25, alpha=0.6, label=geno)
ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("Δφ_xy (°)  — signed step rotation in imaging plane")
ax.set_ylabel("Count")
ax.set_title("Step rotation in XY plane\n(positive = CCW, negative = CW)")
ax.legend()

# Per-NB cumulative XY trajectory
for col_idx, geno in enumerate(genotypes):
    ax = axes[1, col_idx + 1]
    sub = df[df["genotype"] == geno]
    for (fn, nb), grp in sub.groupby(["file_name", "NB_id"]):
        g = grp.sort_values("time_point")
        valid = g[g["cum_phi_xy"].notna()]
        if len(valid) > 1:
            ax.plot(valid["time_point"] - 1, valid["cum_phi_xy"],
                    marker="o", alpha=0.55, linewidth=1.2, markersize=4)
    ax.axhline(0, color="k", linewidth=0.5, linestyle=":")
    ax.set_title(f"{geno}: cumulative XY rotation per NB")
    ax.set_xlabel("Steps from t₁")
    ax.set_ylabel("Cumulative Δφ_xy (°)")

plt.suptitle("Rotation axis directions and signed XY-plane rotation", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Parameterizing the 2D simulation

In a 2D simulation, each division axis has a scalar orientation angle φ and rotates by a signed step Δφ drawn from some distribution. The `delta_phi_xy` values computed above are the most direct empirical estimate of that distribution.

### Mean: μ = 0 or μ = observed?

If the rotation mechanism has no preferred direction, the distribution of step rotations is symmetric around zero and μ = 0 — a pure random walk. If there is a systematic directional bias (the spindle consistently tends to rotate CW or CCW), μ ≠ 0.

Two reasons to be cautious about the observed non-zero means here:

1. **Statistical power is low.** With ~30 mud4 steps and ~120 wt steps, the standard error of the mean is σ/√n. For mud4 that's ≈ 13°, comparable to the observed mean itself — so the mean is not significantly different from zero.
2. **Sign consistency is unknown.** Without knowing each embryo's orientation in the microscope, "CCW" in one recording may be biologically the same direction as "CW" in another, making the pooled mean meaningless as a directional parameter.

**Recommendation:** run simulations with **both** μ = 0 (null model) and μ = observed mean. If the output distributions look similar, the mean is not consequential and you can use μ = 0 for simplicity. If they differ substantially, the sign consistency question deserves investigation with the microscopy metadata.

### Standard deviation: σ

σ is more robust — it quantifies how quickly the spindle diffuses in angle space per step regardless of direction. Estimate it from `std(delta_phi_xy)` per genotype.

In [ ]:
params = []
for geno, grp in df.groupby("genotype"):
    vals = grp["delta_phi_xy"].dropna()
    params.append({
        "genotype": geno,
        "n_steps": len(vals),
        "mean_deg": round(vals.mean(), 2),
        "SE_mean": round(vals.std() / np.sqrt(len(vals)), 2),
        "std_deg": round(vals.std(), 2),
        "median_deg": round(vals.median(), 2),
        "n_large_steps (|Δφ|>90°)": int((vals.abs() > 90).sum()),
    })

sim_params = pd.DataFrame(params).set_index("genotype")
print("Parameters for Normal(μ, σ) step distribution:")
sim_params

In [ ]:
fig, axes = plt.subplots(1, len(genotypes), figsize=(7 * len(genotypes), 4))
if len(genotypes) == 1:
    axes = [axes]

x = np.linspace(-220, 220, 500)
for ax, geno in zip(axes, genotypes):
    vals = df[df["genotype"] == geno]["delta_phi_xy"].dropna()
    mu, sigma = vals.mean(), vals.std()
    ax.hist(vals, bins=20, density=True, alpha=0.55, label="observed")
    ax.plot(x, norm.pdf(x, mu, sigma), "r-", linewidth=2,
            label=f"Normal(μ={mu:.1f}°, σ={sigma:.1f}°)")
    ax.plot(x, norm.pdf(x, 0, sigma), "k--", linewidth=1.5,
            label=f"Normal(μ=0, σ={sigma:.1f}°)")
    ax.set_xlabel("Δφ_xy (°)")
    ax.set_ylabel("Density")
    ax.set_title(geno)
    ax.legend(fontsize=9)
    ax.set_xlim(-220, 220)

fig.suptitle("Step distribution: observed vs Normal fits\n"
             "red = fitted mean  |  black dashed = μ forced to 0")
plt.tight_layout()
plt.show()

## 8. Consistency check: does the step distribution predict the cumulative trajectories?

If each step rotation is i.i.d. Normal(μ, σ), then the cumulative XY rotation after n steps is:

$$\varphi_{\text{cum}}(n) = \sum_{i=1}^{n} \Delta\varphi_i \;\sim\; \text{Normal}\!\left(n\mu,\; \sqrt{n}\,\sigma\right)$$

The distribution **fans out proportionally to √n** — the signature of a random walk. The width of the fan after n steps is fully determined by σ alone (plus a drift set by μ).

We simulate many random walks using the fitted Normal parameters and draw the expected percentile bands, then overlay the observed NB trajectories.

**What to look for:**
- Observed lines mostly **inside** the band → the step distribution is self-consistent; the model is adequate.
- Observed lines **wider** than the band → steps are correlated (the spindle "remembers" its recent direction), or the step distribution has heavier tails than Normal.
- Observed lines **narrower** than the band → there may be a restoring force pulling the spindle back toward a preferred orientation.

We show both μ = observed and μ = 0 side by side to assess whether the mean matters.

In [ ]:
rng = np.random.default_rng(42)
N_SIMS = 4000
MAX_STEPS = 12

fig, axes = plt.subplots(len(genotypes), 2,
                         figsize=(14, 5 * len(genotypes)),
                         sharey="row")
if len(genotypes) == 1:
    axes = axes.reshape(1, 2)

step_counts = np.arange(1, MAX_STEPS + 1)

for row_idx, geno in enumerate(genotypes):
    vals = df[df["genotype"] == geno]["delta_phi_xy"].dropna()
    mu_obs, sigma = vals.mean(), vals.std()

    for col_idx, (mu, model_label) in enumerate([
        (mu_obs, f"μ = {mu_obs:.1f}° (observed mean)"),
        (0.0,    "μ = 0  (null model)"),
    ]):
        ax = axes[row_idx, col_idx]

        # Simulate random walks
        sim = np.cumsum(rng.normal(mu, sigma, (N_SIMS, MAX_STEPS)), axis=1)
        p5, p25, p75, p95 = np.percentile(sim, [5, 25, 75, 95], axis=0)

        ax.fill_between(step_counts, p5, p95, alpha=0.15, color="steelblue",
                        label="5–95th %ile (simulated)")
        ax.fill_between(step_counts, p25, p75, alpha=0.30, color="steelblue",
                        label="25–75th %ile (simulated)")
        ax.plot(step_counts, np.percentile(sim, 50, axis=0),
                "b-", linewidth=1.2, label="simulated median")

        # Observed NB trajectories
        sub = df[df["genotype"] == geno]
        for (fn, nb), grp in sub.groupby(["file_name", "NB_id"]):
            g = grp.sort_values("time_point")
            valid = g[g["cum_phi_xy"].notna()]
            if len(valid) < 2:
                continue
            ax.plot(valid["time_point"] - 1, valid["cum_phi_xy"],
                    "o-", color="tomato", alpha=0.5, linewidth=1.5, markersize=4)

        ax.axhline(0, color="k", linewidth=0.5, linestyle=":")
        ax.set_title(f"{geno} — {model_label}\n(σ = {sigma:.1f}°,  n_steps = {len(vals)})")
        ax.set_xlabel("Steps from first time point")
        ax.set_ylabel("Cumulative XY rotation (°)")
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8, loc="upper left")

fig.suptitle(
    "Consistency check: observed NB trajectories (red) vs simulated random-walk bands (blue)\n"
    "Self-consistent = most red lines fall within the blue band",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## 9. Within-NB directional persistence: do divisions keep going the same way?

The simulation question "should μ = 0?" is really asking whether each step is drawn independently from a zero-mean distribution, or whether there is a systematic tendency. One way to probe this is to ask a simpler, sign-only question:

**After a spindle rotates in one direction at step t, does it tend to continue in that direction at step t+1, or reverse?**

If steps are i.i.d. (the random walk model), consecutive steps should be in the same direction ~50% of the time by chance. More than 50% = **persistence** (positive lag-1 autocorrelation); less than 50% = **oscillation** (negative autocorrelation, like a restoring force).

We compute all consecutive within-NB step pairs from `delta_phi_xy` and look at their lag-1 autocorrelation both as a correlation coefficient and as a simple same/different count.

### Why this matters for simulation

- **Persistent steps (r > 0)**: the i.i.d. model underestimates how far the spindle diffuses. The cumulative distribution fans out faster than √n·σ predicts — you'd need a correlated-step (AR(1)) model instead of pure i.i.d.
- **Oscillating steps (r < 0)**: there is a restoring force. The cumulative distribution is tighter than the random walk predicts.
- **r ≈ 0**: the i.i.d. Normal model is self-consistent and appropriate.

Note: with the small sample sizes here (18 mud4 pairs, 96 wt pairs), the individual r values have wide uncertainty. The per-NB sign sequences (Panel 3) help make the pattern visible at the individual cell level.

In [ ]:
# Build all consecutive step pairs within each NB
pairs = []
for (sf, fn, nb), grp in df.groupby(["source_file", "file_name", "NB_id"]):
    geno = grp["genotype"].iloc[0]
    g = grp[grp["delta_phi_xy"].notna()].sort_values("time_point")
    sv = g["delta_phi_xy"].values
    tps = g["time_point"].values
    for i in range(len(sv) - 1):
        if tps[i + 1] == tps[i] + 1:  # only truly consecutive time points
            pairs.append({
                "genotype": geno, "file_name": fn, "NB_id": nb,
                "step_t": sv[i], "step_t1": sv[i + 1],
            })

pairs_df = pd.DataFrame(pairs)
pairs_df["same_dir"] = (pairs_df["step_t"] * pairs_df["step_t1"]) > 0

# Per-NB: fraction of consecutive pairs in same direction
nb_persist = (
    pairs_df.groupby(["genotype", "file_name", "NB_id"])
    .agg(n_pairs=("same_dir", "count"), frac_same=("same_dir", "mean"))
    .reset_index()
)

print(f"Total consecutive step pairs: {len(pairs_df)}")
print()
print("Fraction of same-direction pairs by genotype:")
print(pairs_df.groupby("genotype")["same_dir"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "n_same", "count": "n_total", "mean": "frac_same"}))
print()
print("Per-NB persistence (NBs with ≥2 steps only):")
print(nb_persist.groupby("genotype")["frac_same"].describe().round(2))


In [ ]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: scatter of step[t] vs step[t+1] ---
ax = axes[0]
colors = {"mud4": "C1", "wt": "C0"}
for geno, grp in pairs_df.groupby("genotype"):
    ax.scatter(grp["step_t"], grp["step_t1"],
               color=colors.get(geno, "gray"), alpha=0.6, s=40,
               label=f"{geno} (n={len(grp)})")

# Quadrant shading: Q1+Q3 (same direction) in light green
xlim = max(pairs_df["step_t"].abs().max(), pairs_df["step_t1"].abs().max()) * 1.15
ax.fill_between([-xlim, 0], 0, xlim, alpha=0.07, color="green")
ax.fill_between([0, xlim], -xlim, 0, alpha=0.07, color="green")
ax.axhline(0, color="k", linewidth=0.6, linestyle="--")
ax.axvline(0, color="k", linewidth=0.6, linestyle="--")

# Overall lag-1 correlation
r, p = pearsonr(pairs_df["step_t"], pairs_df["step_t1"])
ax.set_xlabel("Step rotation Δφ at time t (°)")
ax.set_ylabel("Step rotation Δφ at time t+1 (°)")
ax.set_title(f"Lag-1 autocorrelation of XY step rotations\n"
             f"Pooled r = {r:.3f}  (p = {p:.3f})\n"
             f"Green shading = same direction as previous step")
ax.set_xlim(-xlim, xlim); ax.set_ylim(-xlim, xlim)
ax.set_aspect("equal")
ax.legend(fontsize=9)

# Per-genotype r values as text
for geno, grp in pairs_df.groupby("genotype"):
    ri, pi = pearsonr(grp["step_t"], grp["step_t1"])
    ax.annotate(f"{geno}: r={ri:.2f} (p={pi:.2f})",
                xy=(0.03, 0.97 - list(pairs_df["genotype"].unique()).index(geno) * 0.08),
                xycoords="axes fraction", fontsize=8, va="top",
                color=colors.get(geno, "gray"))

# --- Panel 2: per-NB fraction of steps in same direction ---
ax = axes[1]
for geno in genotypes:
    sub = nb_persist[nb_persist["genotype"] == geno]
    ax.scatter(
        [geno] * len(sub), sub["frac_same"],
        color=colors.get(geno, "gray"), alpha=0.7, s=60, zorder=3,
    )
    ax.scatter([geno], [sub["frac_same"].mean()],
               color=colors.get(geno, "gray"), s=120, marker="D",
               edgecolors="k", linewidths=1.2, zorder=4,
               label=f"{geno} mean = {sub['frac_same'].mean():.2f}")

ax.axhline(0.5, color="k", linewidth=1, linestyle="--", label="0.5 (random)")
ax.set_ylabel("Fraction of consecutive step pairs\nin same rotation direction")
ax.set_xlabel("Genotype  (each dot = one NB)")
ax.set_title("Within-NB directional persistence\n"
             "Above 0.5 = tends to continue same direction\n"
             "Below 0.5 = tends to reverse direction")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8)

# --- Panel 3: step sign sequences per NB (visual) ---
ax = axes[2]
nb_list = nb_persist.sort_values(["genotype","file_name","NB_id"]).reset_index(drop=True)
for row_i, row in nb_list.iterrows():
    nb_steps = pairs_df[
        (pairs_df["file_name"] == row["file_name"]) &
        (pairs_df["NB_id"] == row["NB_id"])
    ].sort_values("step_t")  # sorted by appearance order

    nb_steps_sorted = pairs_df[
        (pairs_df["file_name"] == row["file_name"]) &
        (pairs_df["NB_id"] == row["NB_id"])
    ].reset_index(drop=True)

    # Draw each consecutive pair as same/different indicator
    for col_i, (_, pair_row) in enumerate(nb_steps_sorted.iterrows()):
        color = "green" if pair_row["same_dir"] else "tomato"
        ax.add_patch(plt.Rectangle(
            (col_i + 0.05, row_i + 0.05), 0.9, 0.9,
            color=color, alpha=0.7
        ))

    geno_label = row["genotype"]
    ax.text(-0.3, row_i + 0.5, f"{geno_label} · {row['NB_id']}",
            va="center", ha="right", fontsize=7,
            color=colors.get(geno_label, "gray"))

ax.set_xlim(-3, 8)
ax.set_ylim(-0.3, len(nb_list) + 0.3)
ax.set_xlabel("Consecutive step index (within NB)")
ax.set_title("Step direction for each NB\ngreen = same as previous, red = reversed")
ax.set_xticks(np.arange(8) + 0.5)
ax.set_xticklabels([f"{i+1}→{i+2}" for i in range(8)], fontsize=8)
ax.set_yticks([])
# Draw a divider between genotypes
mud4_rows = nb_list[nb_list["genotype"] == "mud4"].index
if len(mud4_rows):
    ax.axhline(mud4_rows[-1] + 1, color="gray", linewidth=0.8, linestyle=":")

plt.suptitle("Within-NB directional persistence of rotation steps", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
